In [ ]:
Step 1

In [6]:
# Notebook: homeguard_system.ipynb
# Author: PietFALB
# Date: 2026-10-06
# Description: This script controls the HomeGuard system, which monitors and manages home security devices. It provides functionalities to add, remove, and monitor devices, as well as generate reports on their status.


import pandas as pd
import random as pd
import datetime as pd

from datetime import datetime
from random import random
from pandas import DataFrame

In [8]:
def create_sensor(sensor_id, location, sensor_type, current_value, low=35.0, high=80.0):
    return {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "current_value": current_value,
        "low_threshold": low,
        "high_threshold": high,
    }

In [20]:
def create_alert(sensor_id, level, message):
    return {"sensor_id": sensor_id, "level": level, "message": message}

In [ ]:
def init_system():
    return {
        "mode": "armed",  # "armed" or "maintenance"
        "sensors": [
            create_sensor("1", "Living room", "Motion", None),
            create_sensor("2", "Front door", "Lock", None),
            create_sensor("3", "Bedroom", "Smoke", None),
            create_sensor("4", "Kitchen", "Temperature", None),
        ],
        "alerts": [],
    }

In [26]:
import random

VALID_MODES = ("home", "away", "sleep")

def random_value(sensor_type):
    """Generate a random reading that fits the sensor type."""
    if sensor_type == "motion":
        return random.choice([True, False])
    elif sensor_type == "lock":
        return random.choice(["locked", "unlocked"])
    elif sensor_type == "temperature":
        return round(random.uniform(5.0, 45.0), 1)
    elif sensor_type == "smoke":
        return random.random() < 0.15   # smoke is rare: ~15% chance
    else:
        raise ValueError(f"Unknown sensor type: {sensor_type}")

def create_sensor(sensor_id, location, sensor_type, low=None, high=None):
    return {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "current_value": random_value(sensor_type),
        "low_threshold": low,
        "high_threshold": high,
    }

def create_alert(sensor_id, level, message):
    return {"sensor_id": sensor_id, "level": level, "message": message}

def set_mode(system, new_mode):
    if new_mode in VALID_MODES:
        system["mode"] = new_mode
        return True
    print(f"Invalid mode: {new_mode}. Choose from {VALID_MODES}")
    return False

def update_sensors(system):
    """Simulate new readings for every sensor."""
    for sensor in system["sensors"]:
        sensor["current_value"] = random_value(sensor["type"])

def init_system():
    return {
        "mode": random.choice(VALID_MODES),
        "sensors": [
            create_sensor("M1", "Living room", "motion"),
            create_sensor("L1", "Front door", "lock"),
            create_sensor("T1", "Kitchen", "temperature", low=10.0, high=40.0),
            create_sensor("S1", "Bedroom", "smoke"),
        ],
        "alerts": [],
    }

In [27]:
system = init_system()
for s in system["sensors"]:
    print(s)
print("Mode:", system["mode"], "| Alerts:", len(system["alerts"]))

{'id': 'M1', 'location': 'Living room', 'type': 'motion', 'current_value': True, 'low_threshold': None, 'high_threshold': None}
{'id': 'L1', 'location': 'Front door', 'type': 'lock', 'current_value': 'locked', 'low_threshold': None, 'high_threshold': None}
{'id': 'T1', 'location': 'Kitchen', 'type': 'temperature', 'current_value': 14.0, 'low_threshold': 10.0, 'high_threshold': 40.0}
{'id': 'S1', 'location': 'Bedroom', 'type': 'smoke', 'current_value': False, 'low_threshold': None, 'high_threshold': None}
Mode: away | Alerts: 0


In [28]:
def check_sensor(sensor, mode):
    stype = sensor["type"]
    value = sensor["current_value"]
    where = sensor["location"]

    if stype == "smoke":
        if value:
            return "critical", f"Smoke detected in {where}"

    elif stype == "temperature":
        if value > sensor["high_threshold"]:
            return "critical", f"{where} too hot: {value}°C"
        elif value < sensor["low_threshold"]:
            return "warning", f"{where} too cold: {value}°C"

    elif stype == "motion":
        if value:
            if mode == "away":
                return "critical", f"Unexpected motion in {where}"
            elif mode == "sleep":
                return "warning", f"Motion in {where} during sleep mode"
            # in "home" mode, motion is normal

    elif stype == "lock":
        if value == "unlocked":
            if mode == "away":
                return "critical", f"{where} is unlocked while away"
            elif mode == "sleep":
                return "warning", f"{where} is unlocked during sleep mode"
            # in "home" mode, an unlocked door is fine

    return "ok", None

def process_sensors(system):
    for sensor in system["sensors"]:
        level, message = check_sensor(sensor, system["mode"])
        if level != "ok":
            system["alerts"].append(create_alert(sensor["id"], level, message))

In [29]:
system = init_system()
system["sensors"][0]["current_value"] = True        # motion detected
system["sensors"][1]["current_value"] = "unlocked"  # front door unlocked
system["sensors"][2]["current_value"] = 45.0        # kitchen too hot
system["sensors"][3]["current_value"] = True        # smoke detected

for mode in VALID_MODES:
    system["alerts"].clear()
    set_mode(system, mode)
    process_sensors(system)
    print(mode, "->", len(system["alerts"]), "alerts")

home -> 2 alerts
away -> 4 alerts
sleep -> 4 alerts


In [31]:
random.seed() 
system = init_system()
for s in system["sensors"]:
    print(s["id"], s["location"], "->", s["current_value"])
print("Mode:", system["mode"])

M1 Living room -> True
L1 Front door -> locked
T1 Kitchen -> 43.8
S1 Bedroom -> False
Mode: home


In [33]:
import random
import time
from datetime import datetime, timedelta

VALID_MODES = ("HOME", "AWAY", "SLEEP")
COMFORT_MIN, COMFORT_MAX = 65, 78      # HOME-mode comfort band (°F)
ICONS = {"HIGH": "🚨", "MEDIUM": "⚠️", "LOW": "💡"}

event_log = []

def generate_reading(sensor_type, previous=None):
    """Generate a realistic reading. Temperature drifts from the previous value."""
    if sensor_type == "motion":
        return random.random() < 0.3
    elif sensor_type == "lock":
        if previous == "OPENED":
            return "CLOSED" if random.random() < 0.6 else "OPENED"
        return "OPENED" if random.random() < 0.15 else "CLOSED"
    elif sensor_type == "temperature":
        if previous is None:
            return random.randint(62, 78)
        if random.random() < 0.05:                      # occasional spike
            return random.randint(92, 105)
        return max(40, min(110, previous + random.randint(-3, 3)))
    elif sensor_type == "smoke":
        return random.random() < 0.03
    raise ValueError(f"Unknown sensor type: {sensor_type}")

def describe_reading(sensor_type, value, low=None, high=None):
    if sensor_type == "motion":
        return "Motion detected" if value else "No activity"
    elif sensor_type == "lock":
        return value
    elif sensor_type == "temperature":
        status = "Too Hot" if value > high else "Too Cold" if value < low else "Normal"
        return f"{value}°F ({status})"
    elif sensor_type == "smoke":
        return "SMOKE DETECTED" if value else "CLEAR"

def create_alert(level, category, message):
    return {"level": level, "category": category, "message": message}

def process_reading(stype, location, value, low, high, mode):
    """Return an alert dict if this reading needs one in the current mode, else None."""
    if stype == "smoke":
        if value:
            return create_alert("HIGH", "SAFETY", f"Smoke detected in {location}!")

    elif stype == "temperature":
        if value > high:
            return create_alert("HIGH", "SAFETY", f"{location} temperature critical: {value}°F!")
        if value < low:
            return create_alert("MEDIUM", "SAFETY", f"{location} temperature too low: {value}°F!")
        if mode == "HOME" and not (COMFORT_MIN <= value <= COMFORT_MAX):
            feel = "warm" if value > COMFORT_MAX else "cool"
            return create_alert("LOW", "COMFORT", f"{location} is {feel} ({value}°F)")

    elif stype == "motion":
        if value:
            if mode == "AWAY":
                return create_alert("HIGH", "SECURITY", f"Motion detected in {location} while in AWAY mode!")
            if mode == "SLEEP":
                return create_alert("MEDIUM", "SECURITY", f"Motion detected in {location} while in SLEEP mode!")

    elif stype == "lock":
        if value == "OPENED" and mode in ("AWAY", "SLEEP"):
            return create_alert("HIGH", "SECURITY", f"{location} opened while in {mode} mode!")

    return None

def log_event(timestamp, message, show=True):
    """Record every event; print it only when show=True."""
    event_log.append(f"[{timestamp}] {message}")
    if show:
        print(f"[LOG] [{timestamp}] {message}")

def trigger_alert(alert, timestamp):
    icon = ICONS[alert["level"]]
    print(f"[ALERT!] {icon} {alert['level']}: {alert['category']}: {alert['message']}")
    if alert["level"] == "HIGH":
        log_event(timestamp, "Sending notification to homeowner...")
    else:
        log_event(timestamp, "Notification queued for review")

random.seed(1)
for stype in ("motion", "lock", "temperature", "smoke"):
    print(stype, "->", generate_reading(stype))

alert = process_reading("lock", "Front Door", "OPENED", None, None, "AWAY")
print(alert)
trigger_alert(alert, "14:31:00")

motion -> True
lock -> CLOSED
temperature -> 64
smoke -> False
{'level': 'HIGH', 'category': 'SECURITY', 'message': 'Front Door opened while in AWAY mode!'}
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...


In [35]:
class Sensor:
    DEFAULTS = {"motion": False, "lock": "CLOSED", "temperature": 68, "smoke": False}

    def __init__(self, sensor_id, name, location, sensor_type, low=None, high=None):
        self.id = sensor_id
        self.name = name
        self.location = location
        self.type = sensor_type
        self.low = low
        self.high = high
        self.value = self.DEFAULTS[sensor_type]
        self.previous = None

    def read(self):
        """Take a new reading and remember the old one."""
        self.previous = self.value
        self.value = generate_reading(self.type, self.value)
        return self.value

    def isAbnormal(self):
        """True if the reading differs from the resting state (independent of mode)."""
        if self.type in ("motion", "smoke"):
            return bool(self.value)
        if self.type == "lock":
            return self.value == "OPENED"
        return self.value < self.low or self.value > self.high

    def reset(self):
        self.value = self.DEFAULTS[self.type]
        self.previous = None

    def changed(self):
        return self.value != self.previous

    def describe(self):
        return describe_reading(self.type, self.value, self.low, self.high)

    def check(self, mode):
        return process_reading(self.type, self.location, self.value, self.low, self.high, mode)

def build_sensors():
    return [
        Sensor("M1", "Living Room Motion", "Living Room", "motion"),
        Sensor("L1", "Front Door", "Front Door", "lock"),
        Sensor("T1", "Kitchen Temperature", "Kitchen", "temperature", low=50, high=90),
        Sensor("S1", "Bedroom Smoke", "Bedroom", "smoke"),
    ]

random.seed(3)
sensors = build_sensors()
for s in sensors:
    s.read()
    print(f"{s.name}: {s.describe()} | abnormal: {s.isAbnormal()}")

sensors[3].value = True
print("Forced smoke abnormal:", sensors[3].isAbnormal())   # True
sensors[3].reset()
print("After reset:", sensors[3].describe())               # CLEAR

Living Room Motion: Motion detected | abnormal: True
Front Door: CLOSED | abnormal: False
Kitchen Temperature: 69°F (Normal) | abnormal: False
Bedroom Smoke: CLEAR | abnormal: False
Forced smoke abnormal: True
After reset: CLEAR


In [36]:
def run_simulation(mode="AWAY", steps=5, seed=None, delay=0):
    if mode not in VALID_MODES:
        raise ValueError(f"Mode must be one of {VALID_MODES}")
    if seed is not None:
        random.seed(seed)

    sensors = build_sensors()
    event_log.clear()
    clock = datetime.strptime("14:30:00", "%H:%M:%S")
    alert_count = 0

    print("=== HomeGuard Security System ===")
    for tick in range(steps):
        ts = clock.strftime("%H:%M:%S")
        if tick == 0:
            print(f"Time: {ts}")
            print(f"Mode: {mode}\n")
        else:
            print(f"\nTime: {ts}")

        shown = False
        for s in sensors:
            s.read()
            if tick == 0 or s.changed():
                shown = True
                print(f"[READING] {s.name}: {s.describe()}")
                log_event(ts, f"{s.name} = {s.describe()}", show=False)
                alert = s.check(mode)
                if alert:
                    trigger_alert(alert, ts)
                    alert_count += 1
        if not shown:
            print("[READING] No changes")

        clock += timedelta(minutes=1)
        if delay:
            time.sleep(delay)

    print(f"\n=== Simulation complete: {alert_count} alert(s), {len(event_log)} events logged ===")
    return {"alerts": alert_count, "log": list(event_log)}


run_simulation(mode="AWAY", steps=6, seed=7)

=== HomeGuard Security System ===
Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 65°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 63°F (Normal)

Time: 14:32:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...

Time: 14:33:00
[READING] Kitchen Temperature: 64°F (Normal)

Time: 14:34:00
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 61°F (Normal)

Time: 14:35:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [14:35:00] Sending notification to homeowner...
[R

{'alerts': 3,
 'log': ['[14:30:00] Living Room Motion = No activity',
  '[14:30:00] Front Door = CLOSED',
  '[14:30:00] Kitchen Temperature = 65°F (Normal)',
  '[14:30:00] Bedroom Smoke = CLEAR',
  '[14:31:00] Living Room Motion = Motion detected',
  '[14:31:00] Sending notification to homeowner...',
  '[14:31:00] Kitchen Temperature = 63°F (Normal)',
  '[14:32:00] Living Room Motion = No activity',
  '[14:32:00] Front Door = OPENED',
  '[14:32:00] Sending notification to homeowner...',
  '[14:33:00] Kitchen Temperature = 64°F (Normal)',
  '[14:34:00] Front Door = CLOSED',
  '[14:34:00] Kitchen Temperature = 61°F (Normal)',
  '[14:35:00] Living Room Motion = Motion detected',
  '[14:35:00] Sending notification to homeowner...',
  '[14:35:00] Kitchen Temperature = 62°F (Normal)']}

In [37]:
def test_security():
    assert process_reading("lock", "Front Door", "OPENED", None, None, "AWAY")["level"] == "HIGH"
    assert process_reading("motion", "Living Room", True, None, None, "AWAY")["category"] == "SECURITY"
    assert process_reading("motion", "Living Room", True, None, None, "SLEEP")["level"] == "MEDIUM"
    assert process_reading("lock", "Front Door", "OPENED", None, None, "HOME") is None
    assert process_reading("motion", "Living Room", True, None, None, "HOME") is None
    print("Security test passed")

def test_safety():
    for mode in VALID_MODES:   # safety alerts fire in every mode
        assert process_reading("smoke", "Bedroom", True, None, None, mode)["level"] == "HIGH"
        assert process_reading("temperature", "Kitchen", 95, 50, 90, mode)["level"] == "HIGH"
        assert process_reading("temperature", "Kitchen", 45, 50, 90, mode)["level"] == "MEDIUM"
    assert process_reading("smoke", "Bedroom", False, None, None, "AWAY") is None
    print("Safety test passed")

def test_comfort():
    alert = process_reading("temperature", "Kitchen", 82, 50, 90, "HOME")
    assert alert["category"] == "COMFORT" and alert["level"] == "LOW"
    assert process_reading("temperature", "Kitchen", 82, 50, 90, "AWAY") is None   # no comfort alerts away
    assert process_reading("temperature", "Kitchen", 70, 50, 90, "HOME") is None   # inside the band
    print("Comfort test passed")

test_security()
test_safety()
test_comfort()

for mode in VALID_MODES:
    print(f"\n{'#' * 10} {mode} {'#' * 10}")
    run_simulation(mode=mode, steps=8, seed=11)

Security test passed
Safety test passed
Comfort test passed

########## HOME ##########
=== HomeGuard Security System ===
Time: 14:30:00
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] No changes

Time: 14:32:00
[READING] Front Door: OPENED
[READING] Kitchen Temperature: 65°F (Normal)

Time: 14:33:00
[READING] No changes

Time: 14:34:00
[READING] Kitchen Temperature: 62°F (Normal)
[ALERT!] 💡 LOW: COMFORT: Kitchen is cool (62°F)
[LOG] [14:34:00] Notification queued for review

Time: 14:35:00
[READING] Living Room Motion: Motion detected
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 63°F (Normal)
[ALERT!] 💡 LOW: COMFORT: Kitchen is cool (63°F)
[LOG] [14:35:00] Notification queued for review

Time: 14:36:00
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 64°F (Normal)
[ALERT!] 💡 LOW: COMFORT: Kitchen is cool (64°F)
